# RFGen RGBD Range-Doppler (Colab)

This notebook runs directly in Google Colab. It installs `witwin[radar]`, mounts Google Drive, loads `rfgen_rgbd_sequence.npz`, and generates:

- one preview RD frame
- a continuous 30-frame RD sequence
- `rd_maps_db.npy`, `rd_frame_indices.npy`, `rd_axes.npz`, and `rd_video.gif`

Use a GPU runtime if available, but this notebook also runs on CPU with the PyTorch backend.


In [ ]:
!pip -q install --upgrade pip
!pip -q install witwin[radar] drjit rayd slangtorch imageio ipywidgets gdown matplotlib tqdm scipy

import pathlib, site

site_packages = pathlib.Path(site.getsitepackages()[0])
radar_init = site_packages / 'witwin' / 'radar' / '__init__.py'
text = radar_init.read_text(encoding='utf-8')
text = text.replace('from .trace import TraceResult, Tracer\n', '')
text = text.replace("    'Tracer',\n", '')
text = text.replace("    'TraceResult',\n", '')
radar_init.write_text(text, encoding='utf-8')
print('patched:', radar_init)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%matplotlib inline

import math
import pathlib

import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Image, display
from tqdm.auto import tqdm

# Avoid witwin.radar package-level imports here because they pull in trace.py
# and try to initialize Mitsuba CUDA even though this notebook only needs Radar DSP.
from witwin.radar.radar import Radar
from witwin.radar.sigproc.pointcloud import process_rd

torch.set_grad_enabled(False)


In [ ]:
import re
import subprocess

shared_drive_url = 'https://drive.google.com/file/d/16Nkox9DtgHrWUIvxaYTbwejc-w5CLFKo/view?usp=sharing'
use_shared_link = True

mydrive_npz_path = pathlib.Path('/content/drive/MyDrive/rfgen_rgbd_sequence.npz')
download_npz_path = pathlib.Path('/content/rfgen_rgbd_sequence.npz')
output_dir = pathlib.Path('/content/drive/MyDrive/rfgen_rd_outputs')
output_dir.mkdir(parents=True, exist_ok=True)

if use_shared_link:
    match = re.search(r'/d/([A-Za-z0-9_-]+)', shared_drive_url)
    if match is None:
        raise ValueError(f'Could not parse file id from: {shared_drive_url}')
    file_id = match.group(1)
    if not download_npz_path.exists():
        subprocess.run(['gdown', '--fuzzy', f'https://drive.google.com/uc?id={file_id}', '-O', str(download_npz_path)], check=True)
    npz_path = download_npz_path
else:
    npz_path = mydrive_npz_path

if not npz_path.exists():
    raise FileNotFoundError(f'Missing input file: {npz_path}')

print('input:', npz_path)
print('output:', output_dir)


In [ ]:
def sample_grid(height, width, pixel_stride=1, max_points=0):
    ys, xs = np.mgrid[0:height:pixel_stride, 0:width:pixel_stride]
    ys = ys.reshape(-1)
    xs = xs.reshape(-1)
    if max_points > 0 and xs.size > max_points:
        pick = np.linspace(0, xs.size - 1, max_points, dtype=np.int64)
        ys = ys[pick]
        xs = xs[pick]
    return ys.astype(np.int64), xs.astype(np.int64)


def buffer_masks(sampled_masks, buffer_size):
    if sampled_masks is None or buffer_size <= 0 or sampled_masks.shape[0] <= 1:
        return sampled_masks
    buffered = sampled_masks.clone()
    for offset in range(1, buffer_size + 1):
        buffered[offset:] |= sampled_masks[:-offset]
        buffered[:-offset] |= sampled_masks[offset:]
    return buffered


def build_depth_interpolator(
    depths,
    masks,
    fps,
    *,
    device,
    fov_deg=70.0,
    pixel_stride=1,
    max_points=0,
    depth_min=0.10,
    depth_max=20.0,
    zero_fill=True,
    mask_buffer=1,
):
    depths = np.asarray(depths, dtype=np.float32)
    masks = None if masks is None else np.asarray(masks, dtype=bool)

    num_frames, height, width = depths.shape
    ys, xs = sample_grid(height, width, pixel_stride=pixel_stride, max_points=max_points)

    sampled_depths = torch.as_tensor(depths[:, ys, xs], dtype=torch.float32, device=device)
    sampled_masks = None
    if masks is not None:
        if masks.ndim == 2:
            sampled_masks = torch.as_tensor(masks[ys, xs][None, :], dtype=torch.bool, device=device)
        elif masks.ndim == 3:
            sampled_masks = torch.as_tensor(masks[:, ys, xs], dtype=torch.bool, device=device)
        else:
            raise ValueError(f'Expected masks with shape (H,W) or (T,H,W), got {masks.shape}')
        sampled_masks = buffer_masks(sampled_masks, mask_buffer)

    cx = (width - 1) * 0.5
    cy = (height - 1) * 0.5
    fov_rad = math.radians(float(fov_deg))
    fx = (width * 0.5) / math.tan(fov_rad * 0.5)
    fy = fx

    x_ray = (xs.astype(np.float32) - cx) / fx
    y_ray = -(ys.astype(np.float32) - cy) / fy
    z_ray = -np.ones_like(x_ray, dtype=np.float32)
    rays = torch.as_tensor(np.stack([x_ray, y_ray, z_ray], axis=-1), dtype=torch.float32, device=device)

    total_time = (num_frames - 1) / float(fps)

    def interpolator(time_sec):
        clamped_time = min(max(float(time_sec), 0.0), total_time)
        position = clamped_time * float(fps)
        i0 = min(int(math.floor(position)), num_frames - 1)
        i1 = min(i0 + 1, num_frames - 1)
        alpha = float(position - i0)

        d0 = sampled_depths[i0]
        d1 = sampled_depths[i1]
        valid0 = torch.isfinite(d0) & (d0 >= depth_min) & (d0 <= depth_max)
        valid1 = torch.isfinite(d1) & (d1 >= depth_min) & (d1 <= depth_max)

        if zero_fill:
            d0 = torch.where(valid0, d0, d1)
            d1 = torch.where(valid1, d1, d0)

        p0 = rays * d0.unsqueeze(-1)
        p1 = rays * d1.unsqueeze(-1)
        points = p0 * (1.0 - alpha) + p1 * alpha
        depths_interp = d0 * (1.0 - alpha) + d1 * alpha

        valid = torch.isfinite(depths_interp) & (depths_interp >= depth_min) & (depths_interp <= depth_max)
        valid = valid & torch.isfinite(points).all(dim=-1)

        if sampled_masks is not None:
            mask0 = sampled_masks[0 if sampled_masks.shape[0] == 1 else i0]
            mask1 = sampled_masks[0 if sampled_masks.shape[0] == 1 else i1]
            valid = valid & (mask0 | mask1)

        points = points[valid].contiguous()
        intensities = torch.ones(points.shape[0], dtype=torch.float32, device=device)
        return intensities, points

    return interpolator, total_time


def save_rd_gif(rd_stack, ranges, velocities, frame_indices, gif_path, fps=10):
    vmin = float(np.percentile(rd_stack, 5))
    vmax = float(np.percentile(rd_stack, 99))
    images = []
    for i, rd_db in enumerate(rd_stack):
        fig, ax = plt.subplots(figsize=(8, 4), dpi=150)
        im = ax.imshow(
            rd_db,
            extent=[float(ranges[0]), float(ranges[-1]), float(velocities[0]), float(velocities[-1])],
            origin='lower',
            aspect='auto',
            cmap='jet',
            vmin=vmin,
            vmax=vmax,
        )
        ax.set_xlabel('Range (m)')
        ax.set_ylabel('Velocity (m/s)')
        ax.set_title(f'RD Frame {i + 1}/{len(rd_stack)} (radar frame {int(frame_indices[i])})')
        fig.colorbar(im, ax=ax, label='Magnitude (dB)')
        fig.tight_layout()
        fig.canvas.draw()
        image = np.asarray(fig.canvas.buffer_rgba())[..., :3]
        images.append(image)
        plt.close(fig)
    imageio.mimsave(gif_path, images, fps=fps)


In [ ]:
with np.load(npz_path) as data:
    depths = np.asarray(data['depths'], dtype=np.float32)
    masks = np.asarray(data['masks']) if 'masks' in data.files else None
    rgb = np.asarray(data['rgb']) if 'rgb' in data.files else None
    source_fps = float(data['fps']) if 'fps' in data.files else 30.0

if masks is not None and masks.dtype != np.bool_:
    masks = masks > 0

print('rgb:', None if rgb is None else rgb.shape)
print('depths:', depths.shape, depths.dtype, float(depths.min()), float(depths.max()))
print('masks:', None if masks is None else (masks.shape, masks.dtype, int(masks.sum())))
print(f'source_fps: {source_fps:.2f}')

preview_idx = depths.shape[0] // 2
fig, axes = plt.subplots(1, 3 if rgb is not None else 2, figsize=(12, 4))
if rgb is not None:
    axes[0].imshow(rgb[preview_idx])
    axes[0].set_title('RGB')
    axes[0].axis('off')
    depth_ax = axes[1]
    mask_ax = axes[2]
else:
    depth_ax = axes[0]
    mask_ax = axes[1]

depth_ax.imshow(depths[preview_idx], cmap='viridis')
depth_ax.set_title('Depth (m)')
depth_ax.axis('off')

if masks is not None:
    mask_ax.imshow(masks[preview_idx], cmap='gray')
    mask_ax.set_title('Mask')
    mask_ax.axis('off')

plt.tight_layout()


In [ ]:
radar_config = {
    'num_tx': 3,
    'num_rx': 4,
    'fc': 77e9,
    'slope': 60.012,
    'adc_samples': 256,
    'adc_start_time': 6,
    'sample_rate': 4400,
    'idle_time': 7,
    'ramp_end_time': 65,
    'chirp_per_frame': 128,
    'frame_per_second': 10,
    'num_doppler_bins': 128,
    'num_range_bins': 256,
    'num_angle_bins': 64,
    'power': 15,
    'tx_loc': [[0, 0, 0], [4, 0, 0], [2, 1, 0]],
    'rx_loc': [[-6, 0, 0], [-5, 0, 0], [-4, 0, 0], [-3, 0, 0]],
}

rgbd_params = {
    'fov_deg': 70.0,
    'pixel_stride': 1,
    'max_points': 0,
    'depth_min': 0.10,
    'depth_max': 20.0,
    'zero_fill': True,
    'mask_buffer': 1,
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
backend = 'pytorch'
radar = Radar(radar_config, backend=backend, device=device)

print(f'backend={backend} device={radar.device}')
print(rgbd_params)


In [ ]:
interpolator, total_time = build_depth_interpolator(
    depths,
    masks,
    source_fps,
    device=radar.device,
    fov_deg=rgbd_params['fov_deg'],
    pixel_stride=rgbd_params['pixel_stride'],
    max_points=rgbd_params['max_points'],
    depth_min=rgbd_params['depth_min'],
    depth_max=rgbd_params['depth_max'],
    zero_fill=rgbd_params['zero_fill'],
    mask_buffer=rgbd_params['mask_buffer'],
 )

chirp_period = (radar.config.idle_time + radar.config.ramp_end_time) * 1e-6
radar_valid_time = chirp_period * radar.config.num_tx * max(0, radar.config.chirp_per_frame - 1)
max_output_frames = max(0, int(np.floor((total_time - radar_valid_time) * radar.config.frame_per_second)) + 1)
if max_output_frames <= 0:
    raise ValueError('Sequence is too short for one radar frame.')

rd_num_frames = min(30, max_output_frames)
rd_start_frame = max(0, max_output_frames // 2 - rd_num_frames // 2)
rd_frame_indices = np.arange(rd_start_frame, rd_start_frame + rd_num_frames, dtype=int)

print(f'total_time: {total_time:.3f}s')
print(f'total available radar frames: {max_output_frames}')
print(f'continuous radar frames: {rd_frame_indices[0]} ... {rd_frame_indices[-1]}')


In [ ]:
single_frame_idx = int(rd_frame_indices[0])
t0 = single_frame_idx / radar.config.frame_per_second
frame = radar.mimo(interpolator, t0=t0)
rd_db, _, ranges, velocities = process_rd(radar, frame, tx=0, rx=0, static_clutter_removal=True)
rd_db = rd_db[:, :len(ranges)]

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(
    rd_db,
    extent=[float(ranges[0]), float(ranges[-1]), float(velocities[0]), float(velocities[-1])],
    origin='lower',
    aspect='auto',
    cmap='jet',
)
ax.set_title(f'Single-Frame RD (radar frame {single_frame_idx})')
ax.set_xlabel('Range (m)')
ax.set_ylabel('Velocity (m/s)')
fig.colorbar(im, ax=ax, label='Magnitude (dB)')
plt.tight_layout()
plt.show()


In [ ]:
rd_maps = []
ranges = None
velocities = None

for frame_idx in tqdm(rd_frame_indices, desc='Generating RD frames'):
    t0 = int(frame_idx) / radar.config.frame_per_second
    frame = radar.mimo(interpolator, t0=t0)
    rd_db, _, ranges, velocities = process_rd(radar, frame, tx=0, rx=0, static_clutter_removal=True)
    rd_maps.append(rd_db[:, :len(ranges)].astype(np.float32, copy=False))

rd_stack = np.stack(rd_maps, axis=0)
np.save(output_dir / 'rd_maps_db.npy', rd_stack)
np.save(output_dir / 'rd_frame_indices.npy', rd_frame_indices)
np.savez(output_dir / 'rd_axes.npz', ranges=np.asarray(ranges), velocities=np.asarray(velocities))

print('rd_stack:', rd_stack.shape)
print('saved:', output_dir / 'rd_maps_db.npy')


In [ ]:
gif_path = output_dir / 'rd_video.gif'
save_rd_gif(
    rd_stack,
    np.asarray(ranges),
    np.asarray(velocities),
    rd_frame_indices,
    gif_path,
    fps=max(1, min(len(rd_frame_indices), int(round(radar.config.frame_per_second)))),
)
print(gif_path)
display(Image(filename=str(gif_path)))
